# 线程索引与 SIMT 启动

本节讲 SIMT 编程的"定义 → 启动 → 索引映射"完整链路。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 定义：入口函数与辅助函数


In [ ]:
import pypto_pro.language as pl

THREADS = 256
ELEMENTS = 1000

@pl.vector_function(mode="simt")
def affine(value: pl.DT_FP32, scale: pl.DT_FP32, bias: pl.DT_FP32) -> pl.DT_FP32:
    return value * scale + bias          # 辅助函数：当前线程执行，可返回 Scalar

@pl.vector_function(mode="simt", max_threads=THREADS)
def transform(output, input_tensor, count: pl.DT_UINT32,
              scale: pl.DT_FP32, bias: pl.DT_FP32):
    index = pl.simt.block_idx().x * pl.simt.block_dim().x + pl.simt.thread_idx().x
    if index < count:                      # 尾部边界保护
        output[0, index] = affine(input_tensor[0, index], scale, bias)

- **入口函数**：`max_threads=N` 声明单块上限，每个线程执行一次函数体；
- **辅助函数**：不带 max_threads，在当前线程中执行，可返回 Scalar；
- 参数类型可由实参推导；Tensor/Scalar 可写注解，**Tile 参数不支持注解**；
- **入口函数不能在 SIMT 函数中嵌套调用**。


## 2. 启动：外层 Vector 执行域调用线程块


In [ ]:
@pl.jit(auto_mutex=True)
def transform_kernel(
    input_tensor: pl.Tensor[[1, ELEMENTS], pl.DT_FP32],
    output: pl.Tensor[[1, ELEMENTS], pl.DT_FP32],
    count: pl.DT_UINT32, scale: pl.DT_FP32, bias: pl.DT_FP32,
):
    with pl.section_vector():
        transform[THREADS](output, input_tensor, count, scale, bias)

`simt_func[threads](...)`：方括号是**本次调用的实际块尺寸**（1~3 个编译期正整数，乘积 ≤ max_threads ≤ 2048），圆括号按位置传实参。外层 Kernel 负责 Tile/搬运/流水——**本节采用内嵌 SIMT VF 路径**；当前源码还支持 `@pl.jit(mode="simt", max_threads=...)` 的 Host 直接启动，是否进入目标 0.3.0 包需单独核对。


## 3. Host 侧启动与规模计算


In [ ]:
device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)
input_tensor = torch.randn((1, ELEMENTS), dtype=torch.float32, device=device)
output = torch.empty_like(input_tensor)
scale, bias = 2.0, 1.0
blocks = (ELEMENTS + THREADS - 1) // THREADS     # 本例为 4，须在可用核数内
transform_kernel[None, blocks](input_tensor, output, ELEMENTS, scale, bias)
torch.npu.synchronize()
torch.testing.assert_close(output, input_tensor * scale + bias, rtol=1e-6, atol=1e-6)

以上三段按顺序构成待 950 验证的示例。Host 必须保证 count 不超过 Tensor 容量。一般数据量不可无限增加 blocks（受当前 Stream 可用核数限制），应改用第 7 章 grid-stride。

以 1000 元素、THREADS=256、4 个核为例：4 个 Vector 核各调用一次 `transform[256](...)`，各启动一个 256 线程的 Block；最后一个 Block 只有 232 个线程访问数据，其余 24 个被边界判断跳过。

> **易混点**：Host 启动的 `block_dim` 配置的是**逻辑核数**；SIMT 函数内的 `pl.simt.block_dim()` 返回的是 **Thread Block 各维线程数**——同名不同义。


## 4. 索引接口全家福

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">接口</th><th style="text-align:left;">返回</th><th style="text-align:left;">说明</th></tr>
<tr><td><code>pl.simt.grid_dim()</code></td><td>dim3</td><td>Grid 各维的 Block 数（X 维=生效核数，Y/Z=1）</td></tr>
<tr><td><code>pl.simt.block_dim()</code></td><td>dim3</td><td>本 Block 各维线程数</td></tr>
<tr><td><code>pl.simt.block_idx()</code></td><td>dim3</td><td>本 Block 在 Grid 中的坐标</td></tr>
<tr><td><code>pl.simt.thread_idx()</code></td><td>dim3</td><td>本线程在 Block 内坐标</td></tr>
<tr><td><code>pl.simt.linear_thread_idx()</code></td><td>Scalar</td><td>块内按 X 维优先展开的编号 [0, 块内总数)</td></tr>
</table>

**一维全局索引**（最常用模式）：

```python
index = pl.simt.block_idx().x * pl.simt.block_dim().x + pl.simt.thread_idx().x
```

**二维/三维块**用 linear_thread_idx 展开再组合：

```python
dims = pl.simt.block_dim()
threads_per_block = dims.x * dims.y * dims.z
index = pl.simt.block_idx().x * threads_per_block + pl.simt.linear_thread_idx()
```


### 用输出初值发现漏写

本节 1000 元素例子至少需要 4×256 个线程覆盖；若改成 2 核却仍每线程只处理一次，后 488 个元素无人处理。将输出填成 NaN，可以从未写位置直接观察这类错误，而不是让初始化为零掩盖问题。

大输入不应无限增加启动核数。第 7 章用 grid-stride 让每个线程处理 i、i+stride、i+2×stride，stride 使用实际 Grid 的总线程数。无论哪条路径，Host 必须保证 count 不超过静态 Tensor 容量；边界判断只能保护按正确 count 定义的范围。


## 5. 课后练习

1. （填空）`f[8, 8, 4](...)` 启动的 Block 有 ______ 个线程；`linear_thread_idx()` 的取值范围是 ______。
2. （选择）以下哪项是入口函数的特征？
   A. 必须带 `max_threads`，不能被其他 SIMT 函数嵌套调用<br/>
   B. 可以返回 Scalar 给外层 Kernel<br/>
   C. 可以直接调用 `pl.load`
3. （实践）2048 个元素、THREADS=256、`kernel[None, 2]`（2 核生效）：写出全局 index 的计算式，说明每个核/Block 负责的元素区间；有多少线程空转？

In [ ]:
!cat ./answer/06.03_answer.txt

## 6. 本节小结

- 定义：入口（max_threads）/辅助（可返回 Scalar）；启动：**`f[threads](...)` 嵌在 section_vector 内**；
- Host `block_dim`（核数）≠ `simt.block_dim()`（块内线程数）；
- 一维全局索引公式是默认武器；二维/三维用 linear_thread_idx 展开；
- 尾部线程靠**边界判断**跳过——这是 SIMT 的"循环边界检查"直觉。